# reranker_colab.ipynb — 리랭커(BAAI/bge-reranker-v2-m3)를 코랩 런타임에서 돌리기 위한 노트북

배경: 로컬 컴퓨터(RAM 7.4GB)에서는 bge-reranker-v2-m3(568M 파라미터)를 로드하려고 하면
세그멘테이션 폴트로 프로세스가 죽는다(2026-08-13 실측 확인) — 임베딩(e5-small/e5-large)은
디스크 공간을 확보한 뒤 정상 동작했지만, 리랭커는 모델 자체가 더 커서 로컬 RAM으로는
부족한 것으로 보인다.

이 노트북은 VSCode의 Colab 확장으로 이 파일에 연결해서, 무거운 모델(리랭커·임베딩)만
코랩의 더 넉넉한 런타임에서 실행하기 위한 것이다. `agent/` 패키지 코드는 건드리지 않는다 —
같은 코드를 어디서 실행하느냐만 다르게 하는 것이 목표(로컬 코드와 갈라지지 않게).

## 사용법
1. VSCode에서 이 노트북 열기 → 우측 상단 "커널 선택" → "Colab" → "New Colab Server" → 구글 계정 로그인
2. 아래 셀을 순서대로 실행해서 (1) 연결 확인 → (2) 리랭커 로드 확인까지 검증
3. 여기까지 되면, 다음 단계로 실제 배치 파이프라인과 연동하는 방법을 이어서 설계한다

## 1. 연결 확인 — 이게 코랩에서 도는지, 로컬에서 도는지부터 확인

In [9]:
import platform, os
print("플랫폼:", platform.platform())
print("코랩 여부(google.colab 모듈 있는지):", end=" ")
try:
    import google.colab  # noqa: F401
    print("코랩에서 실행 중")
except ImportError:
    print("코랩 아님 — 로컬에서 도는 중 (커널 선택이 안 된 상태일 수 있음)")

# 메모리 확인 (코랩이면 보통 12GB+, Pro면 더 많음)
!cat /proc/meminfo 2>/dev/null | head -3 || echo "(Linux 환경 아님 — Windows 로컬일 가능성)"

플랫폼: Linux-6.6.122+-x86_64-with-glibc2.35
코랩 여부(google.colab 모듈 있는지): 코랩에서 실행 중
MemTotal:       87528512 kB
MemFree:        69376384 kB
MemAvailable:   83459464 kB


## 2. 리랭커(bge-reranker-v2-m3) 로드 테스트
로컬에서 세그멘테이션 폴트로 죽었던 바로 그 모델. 여기서 정상 로드되는지 확인.

In [10]:
!pip install -q sentence-transformers

In [11]:
import time
from sentence_transformers import CrossEncoder

t0 = time.time()
model = CrossEncoder("BAAI/bge-reranker-v2-m3")
print(f"로드 성공! 소요: {time.time() - t0:.1f}초")

score = model.predict([("소매 판매가 늘었다", "소매판매액지수")])
print("테스트 점수:", score)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

로드 성공! 소요: 4.0초
테스트 점수: [0.6988918]


## 3. 실제 파이프라인 연동

로컬에서 아래 스크립트로 1~2단계(분류+claim추출)와 3단계 키워드매칭까지 끝내면
`data/rerank_pending.json`이 생긴다(claim별 `keyword_candidates`만 포함 — 임베딩/VDB
병합은 아직 안 함).

(2026-08-17: 2026-08-15에 임베딩 매칭을 로컬로 옮겼었는데, claim이 여러 건이면(2건만
있어도 재현됨) `batch_embedding_search()` 호출 중 세그폴트로 프로세스가 죽는 게 실측
확인돼서 다시 코랩으로 되돌렸다(아래 3-1 셀).

2026-08-18: VDB(KOSIS 표 28만7천여 개)를 로컬 Chroma에서 Supabase(pgvector)로 옮기면서
코랩에서도 인터넷으로 접근 가능해졌다 — 예전엔 "코랩에서 접근 불가라 건너뛴다"고 했던
VDB를 이제 여기서 같이 조회한다(아래 3-1 셀). 임베딩 모델도 e5-large -> e5-small로
바뀌었다(Supabase 무료 티어 용량 제한 대응, agent/mapping/embedding_search.py의
KOSIS_EMBEDDING_MODEL과 동일해야 함).)

이 파일을 아래 셀에서 업로드하면, 3-1에서 임베딩+VDB 매칭까지 마저 하고, 그다음 리랭커로
점수를 매겨 `rerank_results.json`을 다운로드해준다. 그걸 로컬 `data/` 폴더에 넣고
`python -m agent.pipeline.resume_after_rerank`를 실행하면 4~8단계가 마저 진행된다.

점수 계산 방식(시그모이드 → 순위 기반 verified 승격)은 `agent/mapping/reranker.py`의
`rerank()`/`_promote_verified_within_top_ranks()`랑 똑같이 맞춰뒀다 — 로컬에서 리랭커가
직접 돌 때랑 결과가 갈라지지 않게.

In [12]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

# 아래 SRC 경로를 본인 드라이브에 업로드한 rerank_pending.json 실제 경로로 바꾸세요.
# (구글 드라이브 웹사이트나 드라이브 앱에서 이 파일을 "내 드라이브" 최상위에 끌어다 놓으면
# 기본값 그대로 써도 됩니다.)
SRC = "/content/drive/MyDrive/rerank_pending.json"
pending_filename = "rerank_pending.json"
shutil.copy(SRC, pending_filename)
print("복사됨:", pending_filename)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
복사됨: rerank_pending.json


In [13]:
import json

with open(pending_filename, encoding="utf-8") as f:
    pending = json.load(f)

# 2026-08-16: 아래 리랭킹 셀의 doc_text_for()가 bare `catalog`를 참조하는데, 이 셀이
# pending["catalog"]만 있고 catalog라는 이름 자체를 assign한 적이 없어서
# NameError("catalog")가 났다(실측 확인). 여기서 명시적으로 꺼내둔다.
catalog = pending["catalog"]

print(f"리랭킹 대상 claim {len(pending['items'])}건, 카탈로그 {len(catalog)}개 표")

리랭킹 대상 claim 33건, 카탈로그 64개 표


## 3-1. 임베딩 + VDB 매칭 (로컬에서 세그폴트 나던 부분을 여기서 대신 실행)

2026-08-17: 로컬에서 `batch_embedding_search()`를 여러 claim에 대해 반복 호출하면(카탈로그
임베딩 1회 + claim마다 쿼리 임베딩) 세그폴트로 프로세스가 죽는 게 실측 확인됐다. 그래서 이
표 매칭 단계를 코랩으로 되돌린다.

2026-08-18: VDB(Supabase/pgvector, KOSIS 표 28만7천여 개)가 이제 클라우드에 있어서
코랩에서도 조회 가능해졌다 — 64개 카탈로그 임베딩 매칭과 같은 셀에서 VDB 조회도 같이
한다.

Supabase 연결 문자열은 노트북에 직접 쓰지 않는다(이 파일이 git에 커밋되므로 비밀번호
노출 위험). 원래 코랩 Secrets(열쇠 아이콘)를 쓰려고 했으나, 이 기능은 VSCode용 코랩
확장에서는 아직 지원 안 됨을 확인했다(웹 브라우저에서 직접 열었을 때만 가능) — 그래서
이미 쓰고 있는 "드라이브에 파일 올려두고 읽어오기" 패턴을 그대로 재사용한다.

**사전 준비(최초 1회만)**: 로컬에서 `{"db_url": ".env의 SUPABASE_DB_URL 값"}` 형식의
JSON을 `supabase_config.json`으로 만들어서 구글 드라이브 "내 드라이브" 최상위에
업로드해두세요(이 파일 자체는 git에 올리지 않음 — `.gitignore` 확인).

`pending["catalog"]`에 카탈로그 전체(표별 embedding_text)가 들어있다 — 이걸로 64개 표
임베딩을 만들고, claim마다 코사인 유사도로 후보를 찾은 뒤 keyword_candidates, VDB
후보랑 합친다. 합치는 규칙은 `agent/mapping/reranker.py`의 `_merge_candidates()`와 동일:
keyword_search가 찾은 표는 그대로 두고, 임베딩/VDB로만 찾은 표는 각각
"(embedding-only, unverified)"/"(vdb-only, unverified)"로 표시해서 다음 셀(리랭킹)의
`is_verified()` 판정이 정확히 걸리게 한다. (참고: 2026-08-16에 VERIFIED_BONUS 가산 방식이
버려지고 순위 기반 승격(`_promote_verified_within_top_ranks`)으로 바뀌었으므로, 여기서는
과거 버전과 달리 점수에 보너스를 더하지 않는다 — 다음 셀이 이미 그 로직을 담당한다.)

In [ ]:
!pip install -q sentence-transformers psycopg2-binary

import json as _json
import numpy as np
import psycopg2
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("intfloat/multilingual-e5-small")

table_ids = list(catalog.keys())
passage_texts = ["passage: " + catalog[tid]["embedding_text"] for tid in table_ids]

print(f"카탈로그 {len(table_ids)}개 표 임베딩 생성 중...")
catalog_vecs = embed_model.encode(passage_texts, convert_to_numpy=True, normalize_embeddings=True)
print("완료")

EMBEDDING_TOP_K = 5
VDB_TOP_K = 10
VDB_MIN_SIMILARITY = 0.75
VDB_TABLE_NAME = "kosis_vdb_tables"

# 드라이브에 미리 올려둔 supabase_config.json({"db_url": "..."})을 읽는다 — 노트북
# 파일 자체(git 커밋됨)엔 연결 문자열을 절대 안 남긴다.
with open("/content/drive/MyDrive/supabase_config.json", encoding="utf-8") as f:
    _SUPABASE_DB_URL = _json.load(f)["db_url"]

_vdb_conn = None


def _get_vdb_connection():
    global _vdb_conn
    if _vdb_conn is None or _vdb_conn.closed:
        _vdb_conn = psycopg2.connect(_SUPABASE_DB_URL)
    return _vdb_conn


def embedding_candidates_for(claim_sentence):
    query_vec = embed_model.encode(
        ["query: " + claim_sentence], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    sims = catalog_vecs @ query_vec
    top_idx = np.argsort(-sims)[:EMBEDDING_TOP_K]
    return query_vec, [
        {
            "table_id": table_ids[i],
            "table_name": catalog[table_ids[i]]["table_name"],
            "score": float(sims[i]),
            "required_slots": [],
            "source_meta": "embedding_search model=intfloat/multilingual-e5-small",
        }
        for i in top_idx
    ]


def vdb_candidates_for(query_vec):
    # agent/kosis/query_vdb.py의 batch_query_vdb()와 동일한 로직(코사인 거리 <=>,
    # 유사도=1-거리, 절대 유사도 하한선) — 여기 코랩에서도 같은 VDB(Supabase)를 조회한다.
    conn = _get_vdb_connection()
    with conn.cursor() as cur:
        cur.execute(
            f"""
            select tbl_id, text, embedding <=> %s::vector as distance
            from {VDB_TABLE_NAME}
            order by embedding <=> %s::vector
            limit %s;
            """,
            (query_vec.tolist(), query_vec.tolist(), VDB_TOP_K),
        )
        rows = cur.fetchall()

    candidates = []
    for tbl_id, text, dist in rows:
        similarity = 1.0 - float(dist)
        if similarity < VDB_MIN_SIMILARITY:
            continue
        candidates.append(
            {
                "table_id": tbl_id,
                "table_name": text,
                "score": similarity,
                "required_slots": [],
                "source_meta": "kosis_vdb model=intfloat/multilingual-e5-small",
            }
        )
    return candidates


def merge_candidates(keyword_cands, embedding_cands, vdb_cands):
    # agent/mapping/reranker.py의 _merge_candidates()와 동일한 규칙: keyword_search가
    # 찾은 표는 그대로 신뢰, 임베딩/VDB로만 찾은 표는 각각 "(embedding-only, unverified)"/
    # "(vdb-only, unverified)"로 표시.
    merged = {}
    for c in keyword_cands:
        merged[c["table_id"]] = c
    for c in embedding_cands:
        if c["table_id"] not in merged:
            merged[c["table_id"]] = {**c, "source_meta": f"{c['source_meta']} (embedding-only, unverified)"}
        else:
            existing = merged[c["table_id"]]
            merged[c["table_id"]] = {**existing, "source_meta": f"{existing['source_meta']} | {c['source_meta']}"}
    for c in vdb_cands:
        if c["table_id"] not in merged:
            merged[c["table_id"]] = {**c, "source_meta": f"{c['source_meta']} (vdb-only, unverified)"}
        else:
            existing = merged[c["table_id"]]
            merged[c["table_id"]] = {**existing, "source_meta": f"{existing['source_meta']} | {c['source_meta']}"}
    return list(merged.values())


for i, item in enumerate(pending["items"]):
    query_vec, emb_cands = embedding_candidates_for(item["claim"]["sentence"])
    vdb_cands = vdb_candidates_for(query_vec)
    item["merged_candidates"] = merge_candidates(item["keyword_candidates"], emb_cands, vdb_cands)
    if i % 10 == 0:
        print(f"임베딩+VDB 매칭 진행: {i}/{len(pending['items'])}")

print("임베딩 + VDB 매칭 + 병합 완료")

In [15]:
import math


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


# agent/mapping/reranker.py의 _promote_verified_within_top_ranks와 반드시 같은 로직으로
# 유지 — 로컬 경로랑 코랩 경로의 판정 방식이 갈라지면 안 됨.
#
# 2026-08-14: 기존엔 keyword로 검증된(verified) 후보에 고정 보너스(+0.05)를 점수에
# 더했는데, 리랭커 raw score가 거의 항상 0 근처라(시그모이드 통과 후 후보 간 점수
# 스프레드가 0.003 정도밖에 안 됨) +0.05가 꼴찌 후보도 1등으로 만들어버릴 수 있었다.
# 실제로 keyword_search 오탐("유가"가 "유가증권"과 혼동)으로 무관한 표가 verified
# 보너스 덕에, 리랭커가 이미 훨씬 높게 평가해둔 진짜 정답(unverified)을 이기는 사례가
# 확인됐다(이 표의 raw_rerank_score를 직접 봐도 최하위였다). 그래서 점수에 더하는 대신,
# verified 후보가 순수 리랭킹 순위 상위 VERIFIED_PROMOTION_RANK 안에 들 때만 1등으로
# 승격시키는 방식(RRF류 순위 기반 판단)으로 바꿨다 — 리랭커가 확실히 아니라고 판단한
# 경우(순위가 한참 밀림)까지 verified라는 이유만으로 뒤집지 않는다.
#
# 2026-08-16: 3 -> 5로 넓힘. 실제 배치(28건) 재현 검토에서, 판단불가로 빠진 21건 중 8건
# (38%)이 "정확히 맞는 keyword 검증 표"를 4~5등에 갖고 있었는데도 top-3 기준에 걸려
# 승격을 못 받고 있었다(예: "65세 이상 실업률" claim이 4등의 정답 표 DT_1DA7102S(성/
# 연령별 실업률) 대신, 1~3등의 무관한 VDB 후보들로 판단불가 처리됨). 어차피 후보를
# top-5까지만 유지하므로(reranked[:5]), 5는 "안 갖고 있는 후보"가 아니라 "이미 갖고
# 있는데 못 쓴 후보"를 마저 살리는 자연스러운 상한이다.
VERIFIED_PROMOTION_RANK = 5


def is_verified(c):
    return "unverified" not in (c.get("source_meta") or "")


def promote_verified_within_top_ranks(ranked):
    if not ranked or is_verified(ranked[0]):
        return ranked
    for i, c in enumerate(ranked[:VERIFIED_PROMOTION_RANK]):
        if is_verified(c):
            return [c] + ranked[:i] + ranked[i + 1 :]
    return ranked


def doc_text_for(c):
    # 2026-08-15: merged_candidates엔 이제 64개 카탈로그 후보뿐 아니라 VDB(KOSIS 표
    # 28만7천여 개) 후보도 섞여 있는데, pending["catalog"]엔 64개 카탈로그 정보만 있어서
    # VDB 표 ID로 조회하면 KeyError가 났다(실측 확인). VDB 후보는 자기 자신의 table_name을
    # 이미 candidate dict에 들고 있으니 그걸 폴백으로 쓴다.
    entry = catalog.get(c["table_id"])
    if entry is not None:
        return entry["embedding_text"]
    return c.get("table_name") or c["table_id"]


output_items = []
for i, item in enumerate(pending["items"]):
    claim_sentence = item["claim"]["sentence"]
    candidates = item["merged_candidates"]
    if not candidates:
        output_items.append({"item_id": item["item_id"], "candidates": []})
        continue
    docs = [doc_text_for(c) for c in candidates]

    raw_scores = model.predict([(claim_sentence, d) for d in docs])

    reranked = []
    for c, raw in zip(candidates, raw_scores):
        reranked.append({**c, "score": sigmoid(float(raw)), "raw_rerank_score": float(raw)})
    reranked.sort(key=lambda c: c["score"], reverse=True)
    reranked = promote_verified_within_top_ranks(reranked)

    output_items.append({"item_id": item["item_id"], "candidates": reranked[:5]})
    if i % 10 == 0:
        print(f"리랭킹 진행: {i}/{len(pending['items'])}")

print("리랭킹 완료:", len(output_items), "건")

리랭킹 진행: 0/33
리랭킹 진행: 10/33
리랭킹 진행: 20/33
리랭킹 진행: 30/33
리랭킹 완료: 33 건


In [16]:
import shutil

with open("rerank_results.json", "w", encoding="utf-8") as f:
    json.dump({"items": output_items}, f, ensure_ascii=False, indent=2)

# files.download()는 이 VS Code<->코랩 연결에서는 성공 메시지만 찍히고 실제 브라우저
# 다운로드가 안 뜨는 문제가 있음(업로드 위젯 때와 동일한 제약, 2026-08-14 확인).
# 이미 마운트된 드라이브에 저장해서 드라이브 웹/앱에서 직접 받는 방식으로 우회.
DRIVE_OUT = "/content/drive/MyDrive/rerank_results.json"
shutil.copy("rerank_results.json", DRIVE_OUT)
print(f"저장 완료: {DRIVE_OUT}")
print("구글 드라이브(내 드라이브 최상위)에서 rerank_results.json을 다운로드해서")
print("로컬 data/ 폴더에 옮기고 resume_after_rerank.py를 실행하세요.")

저장 완료: /content/drive/MyDrive/rerank_results.json
구글 드라이브(내 드라이브 최상위)에서 rerank_results.json을 다운로드해서
로컬 data/ 폴더에 옮기고 resume_after_rerank.py를 실행하세요.
